# Architecture A — Colab engineering sample
**ALI ZBEEB · FATIMA KARNIB · HAIDAR MADI**

Frozen AraBEST-RQ → masked temporal mean of final encoder frames → linear 848 → 20.
Select **Runtime → Change runtime type → T4 GPU (or better)**, then run top to bottom.
This example uses the fixed 500/200 pilot for engineering checks. Full training goes
to the training team using `docs/week2_acoustic_A.md`; no full-data result is implied.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, shutil
from datetime import datetime, timezone

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
MODE = 'smoke'  # 'overfit' runs only the training-subset memorization gate
assert MODE in ('overfit', 'smoke')
REPO_ROOT = Path('/content/Arabic-Dialect-Identification-A')


## Fetch the immutable implementation
The code revision is fixed below; do not switch to an unrecorded branch after running.

In [ ]:
PINNED_COMMIT = '485c6cd2b80a52d66db2470d94d87ef55d437e2f'
REPOSITORY_URL = 'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git'
if not (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', PINNED_COMMIT], cwd=REPO_ROOT, check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO_ROOT, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip() == PINNED_COMMIT
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip(), 'Use a clean source checkout'
print('Source commit:', PINNED_COMMIT)


## Install and check the environment
The encoder and Jais use separate environments. This sample runs the real encoder on GPU.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'environments/acoustic.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, speechbrain; print(torch.__version__, speechbrain.__version__); assert torch.cuda.is_available(), "Select a GPU runtime"; print(torch.cuda.get_device_name(0))'], check=True)
env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
for pattern in ('test_acoustic_encoder.py', 'test_acoustic_classifier.py', 'test_week2_acoustic_A.py'):
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern], cwd=REPO_ROOT, env=env, check=True)


## Reuse or fetch the 700 pilot WAVs
Existing prepared audio on Drive is reused. Missing pilot clips are fetched from the pinned public dataset.

In [ ]:
REVISION = 'c8da454c9f474bb075d3fe608940ee29e8ccfa43'
AUDIO_ROOT = DATA_ROOT / 'audio' / 'adi20_micro' / REVISION
subprocess.run([sys.executable, '-u', str(REPO_ROOT / 'src/data/fetch_pilot_audio.py'),
                '--audio-root', str(AUDIO_ROOT)], cwd=REPO_ROOT, check=True)


## Run A
The encoder is frozen and its state is checked. The head uses the fixed seed 11, cross-entropy, and the last epoch. Exit 2 is a partial result with explicit failures; keep its evidence and diagnose it.

In [ ]:
RUNS_ROOT = DATA_ROOT / 'runs' / 'week2_acoustic_A_colab'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
env.update(DATA_ROOT=str(DATA_ROOT), AUDIO_ROOT=str(AUDIO_ROOT),
           OUTPUT_ROOT=str(DATA_ROOT / 'outputs' / 'week2_acoustic_A'),
           CACHE_ROOT='/content/huggingface-acoustic-A', RUNS_ROOT=str(RUNS_ROOT),
           DEVICE='cuda', PYTHON=sys.executable)
code = subprocess.call(['bash', 'scripts/run_acoustic_A.sh', MODE], cwd=REPO_ROOT, env=env)
print('Exit code:', code)
assert code == 0, 'Inspect the saved logs and failure/gate records before continuing'


## Inspect and download the run
Send back the ZIP. This run is an engineering sample; it must not be reported as full ADI-20 performance.

In [ ]:
runs = sorted([p for p in RUNS_ROOT.iterdir() if p.is_dir() and p.name.startswith(MODE + '-')], key=lambda p: p.stat().st_mtime)
assert runs, 'No run folder was produced'
run = runs[-1]
record = json.loads((run / 'run_record.json').read_text())
print('Run:', run)
print('Role:', record['result_role'])
print('Frozen encoder:', record['encoder']['frozen'])
print('Gate:', record['overfit_check'])
print('Metrics:', record.get('development_evaluation'))
print('Resources:', record['resources'])
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(run / 'pip-freeze.txt').open('w'), check=True)
console = Path(str(run) + '.console.log')
if console.is_file(): shutil.copy2(console, run / 'console.log')
archive = shutil.make_archive('/content/acoustic_A_' + run.name, 'zip', run)
from google.colab import files
files.download(archive)
